In [ ]:
#!/usr/bin/env python
# coding: utf-8
"""
=====================================================================================
PRODUCTION-READY QUANTAMENTAL EQUITY MODEL USING BLOOMBERG DATA
=====================================================================================
This script:
  • Retrieves index constituents via Bloomberg BDS (using the correct "tickers" parameter)
  • Downloads historical daily data via Bloomberg BDH for key fields (PX_LAST, PX_VOLUME, EQY_DVD_YLD, PX_TO_BOOK_RATIO,
    FWD_PE, EPS_EST, EPS_CONSENSUS, SHORT_INT, OPT_VOLUME, etc.)
  • Retrieves static classification data via Bloomberg BDP (GICS_SECTOR, GICS_INDUSTRY_GROUP, COUNTRY)
  • Computes basic factors:
         Carry = EQY_DVD_YLD
         Momentum = (PX_LAST[t - MOM_SKIP] / PX_LAST[t - MOM_LOOKBACK]) - 1
         Value = 1 / PX_TO_BOOK_RATIO
  • Normalizes signals cross-sectionally and builds a composite signal
  • Resamples daily data to monthly (using month-end values)
  • Constructs a long-short portfolio (top decile long, bottom decile short) and computes performance metrics
  • Produces cumulative PnL plots and ranking tables by sector for research publishing
=====================================================================================
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import datetime
import logging
from xbbg import blp

logging.basicConfig(level=logging.INFO)

# --------------------------
# 0. CONFIGURATION PARAMETERS
# --------------------------
INDEX_MAP = {
    'US': 'SPX Index',       # S&P 500
    'EU': 'SX5E Index',      # EuroStoxx 50
    'CHINA': 'XIN9I Index',  # MSCI China (example)
}

START_DATE = "2010-01-01"
END_DATE   = "2025-03-13"
RAW_FREQ   = "D"         # Daily data
REBAL_FREQ = "M"         # Monthly rebalancing

TOP_BOTTOM_PCT = 0.1     # Top/bottom 10%
MIN_LIQUIDITY  = 1e5      # Minimum daily volume

# Momentum parameters (in trading days)
MOM_LOOKBACK = 252       # 12-month lookback
MOM_SKIP     = 21        # Skip the most recent month

TRANSACTION_COST_BPS = 0.1  # Transaction cost in basis points

# --------------------------
# 1. DATA RETRIEVAL FUNCTIONS (using xbbg)
# --------------------------
def get_index_members(index_ticker):
    """
    Retrieve current index constituents via Bloomberg BDS.
    Field used: 'INDX_MEMBERS'
    """
    # Use the correct parameter name "tickers"
    members = blp.bds(tickers=index_ticker, flds='INDX_MEMBERS')
    members = members.rename(columns={'member_ticker_and_exchange_code': 'ticker'})
    members['ticker'] = members['ticker'].str.strip() + " Equity"
    members['index'] = index_ticker
    return members

def transform_bloomberg_data(data):
    """
    Transform Bloomberg data from MultiIndex format to a tidy format.
    Handles the pandas stack() deprecation warning by using melt() instead.
    """
    # Reset index to get date as a column
    df = data.reset_index()
    
    # Get the column names without MultiIndex
    date_col = df.columns[0]  # Usually 'date'
    ticker_field_cols = [(t, f) for t, f in df.columns[1:]]
    tickers = list(set(t for t, _ in ticker_field_cols))
    fields = list(set(f for _, f in ticker_field_cols))
    
    # Create a new DataFrame with proper structure
    result = pd.DataFrame()
    result['date'] = df[date_col].repeat(len(tickers))
    result['ticker'] = np.tile(tickers, len(df))
    
    # Add each field as a column
    for field in fields:
        field_data = []
        for ticker in tickers:
            if (ticker, field) in df.columns:
                field_data.extend(df[(ticker, field)])
            else:
                field_data.extend([np.nan] * len(df))
        result[field] = field_data
    
    return result

def get_equity_data(tickers, start, end, fields=None):
    """
    Retrieve historical daily data for a list of tickers using Bloomberg BDH.
    Default fields:
      PX_LAST, PX_VOLUME, EQY_DVD_YLD, PX_TO_BOOK_RATIO, FWD_PE, EPS_EST, EPS_CONSENSUS, SHORT_INT, OPT_VOLUME
    """
    if fields is None:
        fields = ["PX_LAST", "PX_VOLUME", "EQY_DVD_YLD_IND", "PX_TO_BOOK_RATIO",
                  "BEST_PE_RATIO", "BEST_EPS", "SHORT_INT", "TOT_OPT_VOLUME_CUR_DAY"]
    data = blp.bdh(tickers=tickers, flds=fields, start_date=start, end_date=end)
    data = transform_bloomberg_data(data)
    return data

def get_classification_data(tickers):
    """
    Retrieve static classification data via Bloomberg BDP.
    Fields: GICS_SECTOR, GICS_INDUSTRY_GROUP, COUNTRY.
    """
    fields = ["GICS_SECTOR", "GICS_INDUSTRY_GROUP", "COUNTRY"]
    class_data = blp.bdp(tickers=tickers, flds=fields)
    class_data = class_data.reset_index().rename(columns={'index': 'ticker'})
    return class_data

def build_universe(regions, start, end):
    """
    Build the equity universe:
      1. For each region, get index constituents using BDS.
      2. Union the tickers.
      3. Retrieve historical daily data and classification data.
    Returns:
      equity_data: DataFrame of daily data merged with classification data.
      membership: DataFrame of index membership.
    """
    all_members = []
    for reg in regions:
        idx_tkr = INDEX_MAP[reg.upper()]
        mem = get_index_members(idx_tkr)
        mem['region'] = reg.upper()
        all_members.append(mem)
    membership = pd.concat(all_members, ignore_index=True)
    universe = membership['ticker'].unique().tolist()
    equity_data = get_equity_data(universe, start, end)
    class_data = get_classification_data(universe)
    equity_data = equity_data.merge(class_data, on='ticker', how='left')
    return equity_data, membership

# --------------------------
# 2. FACTOR COMPUTATION FUNCTIONS
# --------------------------
def compute_basic_factors(df):
    """
    Compute basic factors:
      - Carry: EQY_DVD_YLD
      - Momentum: (PX_LAST[t - MOM_SKIP] / PX_LAST[t - MOM_LOOKBACK]) - 1
      - Value: 1 / PX_TO_BOOK_RATIO
    """
    df = df.sort_values(['ticker','date']).reset_index(drop=True)
    df['carry'] = df['EQY_DVD_YLD']
    df['price_skip'] = df.groupby('ticker')['PX_LAST'].shift(MOM_SKIP)
    df['price_lb']   = df.groupby('ticker')['PX_LAST'].shift(MOM_LOOKBACK)
    df['momentum'] = np.where(df['price_lb'].notna(), df['price_skip'] / df['price_lb'] - 1, np.nan)
    df.drop(['price_skip','price_lb'], axis=1, inplace=True)
    df['value'] = 1.0 / df['PX_TO_BOOK_RATIO'].replace(0, np.nan)
    return df

def compute_additional_signals(df):
    """
    Compute additional quantamental signals.
    Replace the placeholders (set as np.nan or simple proxies) with your proprietary calculations.
    """
    # CTA_Momentum: for example, reuse basic momentum
    df['CTA_Momentum'] = df['momentum']
    
    # Regime_Probability: TODO – implement using macro data; placeholder set to NaN
    df['Regime_Probability'] = np.nan
    
    # Crowding: Use SHORT_INT normalized by an assumed float (assume float = 1e6)
    df['Crowding'] = df['SHORT_INT'] / 1e6 if 'SHORT_INT' in df.columns else np.nan
    
    # EPS_Growth: EPS_EST minus EPS_CONSENSUS
    df['EPS_Growth'] = df['EPS_EST'] - df['EPS_CONSENSUS'] if ('EPS_EST' in df.columns and 'EPS_CONSENSUS' in df.columns) else np.nan
    
    # IR_Momentum: TODO – requires bond yield data; placeholder
    df['IR_Momentum'] = np.nan
    
    # Risk_Targeting: inverse of 30-day rolling volatility of PX_LAST returns
    df['ret'] = df.groupby('ticker')['PX_LAST'].pct_change()
    df['vol_30'] = df.groupby('ticker')['ret'].rolling(window=30, min_periods=10).std().reset_index(0, drop=True)
    df['Risk_Targeting'] = (1.0 / df['vol_30']).replace([np.inf, -np.inf], np.nan)
    
    # UpDown_Vol: TODO – implement your proprietary method; placeholder
    df['UpDown_Vol'] = np.nan
    
    # Options_Volumes: OPT_VOLUME / PX_VOLUME
    df['Options_Volumes'] = df['OPT_VOLUME'] / df['PX_VOLUME'] if 'OPT_VOLUME' in df.columns else np.nan
    
    # Adjusted_fPE: Use FWD_PE as proxy (adjust as needed)
    df['Adjusted_fPE'] = df['FWD_PE'] if 'FWD_PE' in df.columns else np.nan
    
    # Sector_PMI: TODO – requires external sector PMI data; placeholder
    df['Sector_PMI'] = np.nan
    
    # Earnings_Revisions: 5-day difference in EPS_EST
    df['Earnings_Revisions'] = df.groupby('ticker')['EPS_EST'].diff(5) if 'EPS_EST' in df.columns else np.nan
    
    df.drop(['ret', 'vol_30'], axis=1, inplace=True)
    return df

def cross_sectional_zscore(df, col, groupby='date'):
    """
    For each date, compute the z-score of a column.
    """
    def zfunc(x):
        mu = x.mean()
        sigma = x.std()
        return (x - mu) / sigma if sigma > 1e-12 else np.nan
    df[col + "_z"] = df.groupby(groupby)[col].transform(zfunc)
    return df

def normalize_signals(df, signals, groupby='date'):
    """
    Normalize a list of signals cross-sectionally by date.
    """
    for signal in signals:
        df = cross_sectional_zscore(df, signal, groupby=groupby)
    return df

def build_composite_signal(df):
    """
    Build a composite signal as the average of the z-scores of the selected signals.
    """
    signals = ['carry', 'momentum', 'value', 'CTA_Momentum', 'Regime_Probability',
               'Crowding', 'EPS_Growth', 'IR_Momentum', 'Risk_Targeting',
               'UpDown_Vol', 'Options_Volumes', 'Adjusted_fPE', 'Sector_PMI', 'Earnings_Revisions']
    df = normalize_signals(df, signals, groupby='date')
    zcols = [s + "_z" for s in signals if s + "_z" in df.columns]
    df['composite'] = df[zcols].mean(axis=1).round(2)
    return df

# --------------------------
# 3. RESAMPLING AND PORTFOLIO CONSTRUCTION
# --------------------------
def resample_to_monthly(df):
    """
    Resample daily data to monthly using month-end values.
    """
    df['date'] = pd.to_datetime(df['date'])
    monthly_list = []
    for tk, sub in df.groupby('ticker'):
        sub = sub.set_index('date').sort_index()
        sub_month = sub.resample('M').last()
        sub_month['ticker'] = tk
        monthly_list.append(sub_month)
    df_month = pd.concat(monthly_list).reset_index()
    df_month = df_month.sort_values(['ticker', 'date']).reset_index(drop=True)
    return df_month

def construct_long_short_portfolio(df, top_pct=TOP_BOTTOM_PCT, min_vol=MIN_LIQUIDITY):
    """
    For each month, rank stocks by composite signal and form a long-short portfolio.
    Long: top decile; Short: bottom decile.
    Compute next month forward return.
    """
    df = df.sort_values(['ticker', 'date']).reset_index(drop=True)
    df['px_next'] = df.groupby('ticker')['PX_LAST'].shift(-1)
    df['ret_fwd'] = df['px_next'] / df['PX_LAST'] - 1.0
    df['yyyymm'] = df['date'].dt.to_period('M')
    
    port_returns = []
    for period, sub in df.groupby('yyyymm'):
        sub = sub[sub['PX_VOLUME'] >= min_vol].dropna(subset=['composite', 'ret_fwd'])
        n = len(sub)
        if n < 30:
            continue
        sub = sub.copy()
        sub['rank'] = sub['composite'].rank(method='first', ascending=False)
        cutoff_top = int(np.floor(top_pct * n))
        cutoff_bot = int(np.floor(top_pct * n))
        sub['position'] = 0.0
        sub.loc[sub['rank'] <= cutoff_top, 'position'] = 1.0
        sub.loc[sub['rank'] > (n - cutoff_bot), 'position'] = -1.0
        num_long = (sub['position'] == 1.0).sum()
        num_short = (sub['position'] == -1.0).sum()
        if num_long > 0:
            sub.loc[sub['position'] == 1.0, 'weight'] = 1.0 / num_long
        if num_short > 0:
            sub.loc[sub['position'] == -1.0, 'weight'] = -1.0 / num_short
        port_ret = (sub['weight'] * sub['ret_fwd']).sum()
        # Apply a simple transaction cost adjustment (assuming turnover factor ~2)
        tc = TRANSACTION_COST_BPS / 10000.0 * 2
        net_ret = port_ret - tc
        port_returns.append({
            'yyyymm': period,
            'n_stocks': n,
            'num_long': num_long,
            'num_short': num_short,
            'port_ret': net_ret
        })
    df_port = pd.DataFrame(port_returns).sort_values('yyyymm').reset_index(drop=True)
    df_port['cum_pnl'] = (1 + df_port['port_ret']).cumprod()
    return df_port

# --------------------------
# 4. RANKING TABLES BY SECTOR
# --------------------------
def build_ranking_tables(df, top_n=3, rating_filter=None):
    """
    Build top and bottom ranking tables by sector.
    Optionally filter by UBS_Rating if provided.
    """
    if rating_filter is not None and 'UBS_Rating' in df.columns:
        df = df[df['UBS_Rating'].isin(rating_filter)].copy()
    grouped = df.groupby('GICS_SECTOR', group_keys=False)
    top_table = grouped.apply(lambda g: g.nlargest(top_n, 'composite'))
    top_table = top_table.sort_values(['GICS_SECTOR', 'composite'], ascending=[True, False])
    bottom_table = grouped.apply(lambda g: g.nsmallest(top_n, 'composite'))
    bottom_table = bottom_table.sort_values(['GICS_SECTOR', 'composite'], ascending=[True, True])
    return top_table, bottom_table

def print_ranking_table(df, title=""):
    """
    Print ranking table.
    """
    cols = ["ticker", "GICS_SECTOR", "GICS_INDUSTRY_GROUP", "COUNTRY", "UBS_Rating",
            "composite", "carry_z", "momentum_z", "value_z",
            "CTA_Momentum_z", "Regime_Probability_z", "Crowding_z", "EPS_Growth_z",
            "IR_Momentum_z", "Risk_Targeting_z", "UpDown_Vol_z", "Options_Volumes_z",
            "Adjusted_fPE_z", "Sector_PMI_z", "Earnings_Revisions_z"]
    exist_cols = [c for c in cols if c in df.columns]
    print(f"\n=== {title} ===\n")
    print(df[exist_cols].to_string(index=False))

# --------------------------
# 5. MAIN WORKFLOW
# --------------------------
def run_quantamental_model(regions=['US', 'EU', 'CHINA']):
    logging.info("Step 1: Building Universe from Bloomberg...")
    eq_data, membership = build_universe(regions, START_DATE, END_DATE)
    
    logging.info("Step 2: Computing Basic Factors...")
    eq_data = compute_basic_factors(eq_data)
    
    logging.info("Step 3: Computing Additional Signals...")
    eq_data = compute_additional_signals(eq_data)
    
    logging.info("Step 4: Building Composite Signal...")
    eq_data = build_composite_signal(eq_data)
    
    logging.info("Step 5: Resampling daily data to monthly...")
    df_monthly = resample_to_monthly(eq_data)
    
    logging.info("Step 6: Constructing monthly long-short portfolio...")
    df_port = construct_long_short_portfolio(df_monthly, top_pct=TOP_BOTTOM_PCT, min_vol=MIN_LIQUIDITY)
    
    # Compute performance metrics
    mrets = df_port['port_ret'].dropna().values
    mean_m = np.mean(mrets)
    std_m = np.std(mrets)
    ann_ret = (1 + mean_m) ** 12 - 1
    ann_vol = std_m * np.sqrt(12)
    ann_sr = ann_ret / (ann_vol + 1e-12)
    logging.info(f"Annualized Return: {ann_ret*100:.2f}% | Annualized Vol: {ann_vol*100:.2f}% | Sharpe: {ann_sr:.2f}")
    
    # Build ranking tables by sector (using Bloomberg classification data)
    top_tbl, bottom_tbl = build_ranking_tables(df_monthly, top_n=3, rating_filter=["Buy", "Neutral"])
    print_ranking_table(top_tbl, title="Top 3 Stocks per Sector (Buy/Neutral)")
    print_ranking_table(bottom_tbl, title="Bottom 3 Stocks per Sector (Buy/Neutral)")
    
    return df_monthly, df_port

# --------------------------
# 6. EXECUTION
# --------------------------
if __name__ == "__main__":
    regions = ['US', 'EU', 'CHINA']
    signals_df, perf_df = run_quantamental_model(regions)
    
    # Plot cumulative PnL
    perf_df['date'] = perf_df['yyyymm'].apply(lambda x: x.to_timestamp('M'))
    plt.figure(figsize=(10, 6))
    plt.plot(perf_df['date'], perf_df['cum_pnl'], marker='o', label='Quantamental L/S Portfolio')
    plt.title("Cumulative PnL of Quantamental Long-Short Strategy")
    plt.xlabel("Date")
    plt.ylabel("Cumulative Return (Base=1.0)")
    plt.legend()
    plt.tight_layout()
    plt.show()
    
    logging.info("Quantamental model run complete. Check signals_df for stock-level signals and perf_df for portfolio performance.")


INFO:root:Step 1: Building Universe from Bloomberg...


In [ ]:
#!/usr/bin/env python
# coding: utf-8
"""
=====================================================================================
PRODUCTION-READY QUANTAMENTAL EQUITY MODEL USING BLOOMBERG DATA
=====================================================================================
This script:
  • Retrieves index constituents via Bloomberg BDS (using the correct "tickers" parameter)
  • Downloads historical daily data via Bloomberg BDH for key fields (PX_LAST, PX_VOLUME, EQY_DVD_YLD, PX_TO_BOOK_RATIO,
    FWD_PE, EPS_EST, EPS_CONSENSUS, SHORT_INT, OPT_VOLUME, etc.)
  • Retrieves static classification data via Bloomberg BDP (GICS_SECTOR, GICS_INDUSTRY_GROUP, COUNTRY)
  • Computes basic factors:
         Carry = EQY_DVD_YLD
         Momentum = (PX_LAST[t - MOM_SKIP] / PX_LAST[t - MOM_LOOKBACK]) - 1
         Value = 1 / PX_TO_BOOK_RATIO
  • Normalizes signals cross-sectionally and builds a composite signal
  • Resamples daily data to monthly (using month-end values)
  • Constructs a long-short portfolio (top decile long, bottom decile short) and computes performance metrics
  • Produces cumulative PnL plots and ranking tables by sector for research publishing
=====================================================================================
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import datetime
import logging
from xbbg import blp

logging.basicConfig(level=logging.INFO)

# --------------------------
# 0. CONFIGURATION PARAMETERS
# --------------------------
INDEX_MAP = {
    'US': 'SPX Index',       # S&P 500
    'EU': 'SX5E Index',      # EuroStoxx 50
    'CHINA': 'XIN9I Index',  # MSCI China (example)
}

START_DATE = "2010-01-01"
END_DATE   = "2025-03-13"
RAW_FREQ   = "D"         # Daily data
REBAL_FREQ = "M"         # Monthly rebalancing

TOP_BOTTOM_PCT = 0.1     # Top/bottom 10%
MIN_LIQUIDITY  = 1e5      # Minimum daily volume

# Momentum parameters (in trading days)
MOM_LOOKBACK = 252       # 12-month lookback
MOM_SKIP     = 21        # Skip the most recent month

TRANSACTION_COST_BPS = 0.1  # Transaction cost in basis points

# --------------------------
# 1. DATA RETRIEVAL FUNCTIONS (using xbbg)
# --------------------------
def get_index_members(index_ticker):
    """
    Retrieve current index constituents via Bloomberg BDS.
    Field used: 'INDX_MEMBERS'
    """
    # Use the correct parameter name "tickers"
    members = blp.bds(tickers=index_ticker, flds='INDX_MEMBERS')
    members = members.rename(columns={'member_ticker_and_exchange_code': 'ticker'})
    members['ticker'] = members['ticker'].str.strip() + " Equity"
    members['index'] = index_ticker
    return members

def transform_bloomberg_data(data):
    """
    Transform Bloomberg data from MultiIndex format to a tidy format.
    Handles the pandas stack() deprecation warning by using melt() instead.
    """
    # Reset index to get date as a column
    df = data.reset_index()
    
    # Get the column names without MultiIndex
    date_col = df.columns[0]  # Usually 'date'
    ticker_field_cols = [(t, f) for t, f in df.columns[1:]]
    tickers = list(set(t for t, _ in ticker_field_cols))
    fields = list(set(f for _, f in ticker_field_cols))
    
    # Create a new DataFrame with proper structure
    result = pd.DataFrame()
    result['date'] = df[date_col].repeat(len(tickers))
    result['ticker'] = np.tile(tickers, len(df))
    
    # Add each field as a column
    for field in fields:
        field_data = []
        for ticker in tickers:
            if (ticker, field) in df.columns:
                field_data.extend(df[(ticker, field)])
            else:
                field_data.extend([np.nan] * len(df))
        result[field] = field_data
    
    return result

def get_equity_data(tickers, start, end, fields=None):
    """
    Retrieve historical daily data for a list of tickers using Bloomberg BDH.
    Default fields:
      PX_LAST, PX_VOLUME, EQY_DVD_YLD, PX_TO_BOOK_RATIO, FWD_PE, EPS_EST, EPS_CONSENSUS, SHORT_INT, OPT_VOLUME
    """
    if fields is None:
        fields = ["PX_LAST", "PX_VOLUME", "EQY_DVD_YLD_IND", "PX_TO_BOOK_RATIO",
                  "BEST_PE_RATIO", "BEST_EPS", "SHORT_INT", "TOT_OPT_VOLUME_CUR_DAY"]
    data = blp.bdh(tickers=tickers, flds=fields, start_date=start, end_date=end)
    data = transform_bloomberg_data(data)
    return data

def get_classification_data(tickers):
    """
    Retrieve static classification data via Bloomberg BDP.
    Fields: GICS_SECTOR, GICS_INDUSTRY_GROUP, COUNTRY.
    """
    fields = ["GICS_SECTOR", "GICS_INDUSTRY_GROUP", "COUNTRY"]
    class_data = blp.bdp(tickers=tickers, flds=fields)
    class_data = class_data.reset_index().rename(columns={'index': 'ticker'})
    return class_data

def build_universe(regions, start, end):
    """
    Build the equity universe:
      1. For each region, get index constituents using BDS.
      2. Union the tickers.
      3. Retrieve historical daily data and classification data.
    Returns:
      equity_data: DataFrame of daily data merged with classification data.
      membership: DataFrame of index membership.
    """
    all_members = []
    for reg in regions:
        idx_tkr = INDEX_MAP[reg.upper()]
        mem = get_index_members(idx_tkr)
        mem['region'] = reg.upper()
        all_members.append(mem)
    membership = pd.concat(all_members, ignore_index=True)
    universe = membership['ticker'].unique().tolist()
    equity_data = get_equity_data(universe, start, end)
    class_data = get_classification_data(universe)
    equity_data = equity_data.merge(class_data, on='ticker', how='left')
    return equity_data, membership

# --------------------------
# 2. FACTOR COMPUTATION FUNCTIONS
# --------------------------
def compute_basic_factors(df):
    """
    Compute basic factors:
      - Carry: EQY_DVD_YLD
      - Momentum: (PX_LAST[t - MOM_SKIP] / PX_LAST[t - MOM_LOOKBACK]) - 1
      - Value: 1 / PX_TO_BOOK_RATIO
    """
    df = df.sort_values(['ticker','date']).reset_index(drop=True)
    df['carry'] = df['EQY_DVD_YLD']
    df['price_skip'] = df.groupby('ticker')['PX_LAST'].shift(MOM_SKIP)
    df['price_lb']   = df.groupby('ticker')['PX_LAST'].shift(MOM_LOOKBACK)
    df['momentum'] = np.where(df['price_lb'].notna(), df['price_skip'] / df['price_lb'] - 1, np.nan)
    df.drop(['price_skip','price_lb'], axis=1, inplace=True)
    df['value'] = 1.0 / df['PX_TO_BOOK_RATIO'].replace(0, np.nan)
    return df

def compute_additional_signals(df):
    """
    Compute additional quantamental signals.
    Replace the placeholders (set as np.nan or simple proxies) with your proprietary calculations.
    """
    # CTA_Momentum: for example, reuse basic momentum
    df['CTA_Momentum'] = df['momentum']
    
    # Regime_Probability: TODO – implement using macro data; placeholder set to NaN
    df['Regime_Probability'] = np.nan
    
    # Crowding: Use SHORT_INT normalized by an assumed float (assume float = 1e6)
    df['Crowding'] = df['SHORT_INT'] / 1e6 if 'SHORT_INT' in df.columns else np.nan
    
    # EPS_Growth: EPS_EST minus EPS_CONSENSUS
    df['EPS_Growth'] = df['EPS_EST'] - df['EPS_CONSENSUS'] if ('EPS_EST' in df.columns and 'EPS_CONSENSUS' in df.columns) else np.nan
    
    # IR_Momentum: TODO – requires bond yield data; placeholder
    df['IR_Momentum'] = np.nan
    
    # Risk_Targeting: inverse of 30-day rolling volatility of PX_LAST returns
    df['ret'] = df.groupby('ticker')['PX_LAST'].pct_change()
    df['vol_30'] = df.groupby('ticker')['ret'].rolling(window=30, min_periods=10).std().reset_index(0, drop=True)
    df['Risk_Targeting'] = (1.0 / df['vol_30']).replace([np.inf, -np.inf], np.nan)
    
    # UpDown_Vol: TODO – implement your proprietary method; placeholder
    df['UpDown_Vol'] = np.nan
    
    # Options_Volumes: OPT_VOLUME / PX_VOLUME
    df['Options_Volumes'] = df['OPT_VOLUME'] / df['PX_VOLUME'] if 'OPT_VOLUME' in df.columns else np.nan
    
    # Adjusted_fPE: Use FWD_PE as proxy (adjust as needed)
    df['Adjusted_fPE'] = df['FWD_PE'] if 'FWD_PE' in df.columns else np.nan
    
    # Sector_PMI: TODO – requires external sector PMI data; placeholder
    df['Sector_PMI'] = np.nan
    
    # Earnings_Revisions: 5-day difference in EPS_EST
    df['Earnings_Revisions'] = df.groupby('ticker')['EPS_EST'].diff(5) if 'EPS_EST' in df.columns else np.nan
    
    df.drop(['ret', 'vol_30'], axis=1, inplace=True)
    return df

def cross_sectional_zscore(df, col, groupby='date'):
    """
    For each date, compute the z-score of a column.
    """
    def zfunc(x):
        mu = x.mean()
        sigma = x.std()
        return (x - mu) / sigma if sigma > 1e-12 else np.nan
    df[col + "_z"] = df.groupby(groupby)[col].transform(zfunc)
    return df

def normalize_signals(df, signals, groupby='date'):
    """
    Normalize a list of signals cross-sectionally by date.
    """
    for signal in signals:
        df = cross_sectional_zscore(df, signal, groupby=groupby)
    return df

def build_composite_signal(df):
    """
    Build a composite signal as the average of the z-scores of the selected signals.
    """
    signals = ['carry', 'momentum', 'value', 'CTA_Momentum', 'Regime_Probability',
               'Crowding', 'EPS_Growth', 'IR_Momentum', 'Risk_Targeting',
               'UpDown_Vol', 'Options_Volumes', 'Adjusted_fPE', 'Sector_PMI', 'Earnings_Revisions']
    df = normalize_signals(df, signals, groupby='date')
    zcols = [s + "_z" for s in signals if s + "_z" in df.columns]
    df['composite'] = df[zcols].mean(axis=1).round(2)
    return df

# --------------------------
# 3. RESAMPLING AND PORTFOLIO CONSTRUCTION
# --------------------------
def resample_to_monthly(df):
    """
    Resample daily data to monthly using month-end values.
    """
    df['date'] = pd.to_datetime(df['date'])
    monthly_list = []
    for tk, sub in df.groupby('ticker'):
        sub = sub.set_index('date').sort_index()
        sub_month = sub.resample('M').last()
        sub_month['ticker'] = tk
        monthly_list.append(sub_month)
    df_month = pd.concat(monthly_list).reset_index()
    df_month = df_month.sort_values(['ticker', 'date']).reset_index(drop=True)
    return df_month

def construct_long_short_portfolio(df, top_pct=TOP_BOTTOM_PCT, min_vol=MIN_LIQUIDITY):
    """
    For each month, rank stocks by composite signal and form a long-short portfolio.
    Long: top decile; Short: bottom decile.
    Compute next month forward return.
    """
    df = df.sort_values(['ticker', 'date']).reset_index(drop=True)
    df['px_next'] = df.groupby('ticker')['PX_LAST'].shift(-1)
    df['ret_fwd'] = df['px_next'] / df['PX_LAST'] - 1.0
    df['yyyymm'] = df['date'].dt.to_period('M')
    
    port_returns = []
    for period, sub in df.groupby('yyyymm'):
        sub = sub[sub['PX_VOLUME'] >= min_vol].dropna(subset=['composite', 'ret_fwd'])
        n = len(sub)
        if n < 30:
            continue
        sub = sub.copy()
        sub['rank'] = sub['composite'].rank(method='first', ascending=False)
        cutoff_top = int(np.floor(top_pct * n))
        cutoff_bot = int(np.floor(top_pct * n))
        sub['position'] = 0.0
        sub.loc[sub['rank'] <= cutoff_top, 'position'] = 1.0
        sub.loc[sub['rank'] > (n - cutoff_bot), 'position'] = -1.0
        num_long = (sub['position'] == 1.0).sum()
        num_short = (sub['position'] == -1.0).sum()
        if num_long > 0:
            sub.loc[sub['position'] == 1.0, 'weight'] = 1.0 / num_long
        if num_short > 0:
            sub.loc[sub['position'] == -1.0, 'weight'] = -1.0 / num_short
        port_ret = (sub['weight'] * sub['ret_fwd']).sum()
        # Apply a simple transaction cost adjustment (assuming turnover factor ~2)
        tc = TRANSACTION_COST_BPS / 10000.0 * 2
        net_ret = port_ret - tc
        port_returns.append({
            'yyyymm': period,
            'n_stocks': n,
            'num_long': num_long,
            'num_short': num_short,
            'port_ret': net_ret
        })
    df_port = pd.DataFrame(port_returns).sort_values('yyyymm').reset_index(drop=True)
    df_port['cum_pnl'] = (1 + df_port['port_ret']).cumprod()
    return df_port

# --------------------------
# 4. RANKING TABLES BY SECTOR
# --------------------------
def build_ranking_tables(df, top_n=3, rating_filter=None):
    """
    Build top and bottom ranking tables by sector.
    Optionally filter by UBS_Rating if provided.
    """
    if rating_filter is not None and 'UBS_Rating' in df.columns:
        df = df[df['UBS_Rating'].isin(rating_filter)].copy()
    grouped = df.groupby('GICS_SECTOR', group_keys=False)
    top_table = grouped.apply(lambda g: g.nlargest(top_n, 'composite'))
    top_table = top_table.sort_values(['GICS_SECTOR', 'composite'], ascending=[True, False])
    bottom_table = grouped.apply(lambda g: g.nsmallest(top_n, 'composite'))
    bottom_table = bottom_table.sort_values(['GICS_SECTOR', 'composite'], ascending=[True, True])
    return top_table, bottom_table

def print_ranking_table(df, title=""):
    """
    Print ranking table.
    """
    cols = ["ticker", "GICS_SECTOR", "GICS_INDUSTRY_GROUP", "COUNTRY", "UBS_Rating",
            "composite", "carry_z", "momentum_z", "value_z",
            "CTA_Momentum_z", "Regime_Probability_z", "Crowding_z", "EPS_Growth_z",
            "IR_Momentum_z", "Risk_Targeting_z", "UpDown_Vol_z", "Options_Volumes_z",
            "Adjusted_fPE_z", "Sector_PMI_z", "Earnings_Revisions_z"]
    exist_cols = [c for c in cols if c in df.columns]
    print(f"\n=== {title} ===\n")
    print(df[exist_cols].to_string(index=False))

# --------------------------
# 5. MAIN WORKFLOW
# --------------------------
def run_quantamental_model(regions=['US', 'EU', 'CHINA']):
    logging.info("Step 1: Building Universe from Bloomberg...")
    eq_data, membership = build_universe(regions, START_DATE, END_DATE)
    
    logging.info("Step 2: Computing Basic Factors...")
    eq_data = compute_basic_factors(eq_data)
    
    logging.info("Step 3: Computing Additional Signals...")
    eq_data = compute_additional_signals(eq_data)
    
    logging.info("Step 4: Building Composite Signal...")
    eq_data = build_composite_signal(eq_data)
    
    logging.info("Step 5: Resampling daily data to monthly...")
    df_monthly = resample_to_monthly(eq_data)
    
    logging.info("Step 6: Constructing monthly long-short portfolio...")
    df_port = construct_long_short_portfolio(df_monthly, top_pct=TOP_BOTTOM_PCT, min_vol=MIN_LIQUIDITY)
    
    # Compute performance metrics
    mrets = df_port['port_ret'].dropna().values
    mean_m = np.mean(mrets)
    std_m = np.std(mrets)
    ann_ret = (1 + mean_m) ** 12 - 1
    ann_vol = std_m * np.sqrt(12)
    ann_sr = ann_ret / (ann_vol + 1e-12)
    logging.info(f"Annualized Return: {ann_ret*100:.2f}% | Annualized Vol: {ann_vol*100:.2f}% | Sharpe: {ann_sr:.2f}")
    
    # Build ranking tables by sector (using Bloomberg classification data)
    top_tbl, bottom_tbl = build_ranking_tables(df_monthly, top_n=3, rating_filter=["Buy", "Neutral"])
    print_ranking_table(top_tbl, title="Top 3 Stocks per Sector (Buy/Neutral)")
    print_ranking_table(bottom_tbl, title="Bottom 3 Stocks per Sector (Buy/Neutral)")
    
    return df_monthly, df_port

# --------------------------
# 6. EXECUTION
# --------------------------
if __name__ == "__main__":
    regions = ['US', 'EU', 'CHINA']
    signals_df, perf_df = run_quantamental_model(regions)
    
    # Plot cumulative PnL
    perf_df['date'] = perf_df['yyyymm'].apply(lambda x: x.to_timestamp('M'))
    plt.figure(figsize=(10, 6))
    plt.plot(perf_df['date'], perf_df['cum_pnl'], marker='o', label='Quantamental L/S Portfolio')
    plt.title("Cumulative PnL of Quantamental Long-Short Strategy")
    plt.xlabel("Date")
    plt.ylabel("Cumulative Return (Base=1.0)")
    plt.legend()
    plt.tight_layout()
    plt.show()
    
    logging.info("Quantamental model run complete. Check signals_df for stock-level signals and perf_df for portfolio performance.")


In [ ]:
start_date = (datetime.datetime.now() - datetime.timedelta(days=2*365)).strftime('%Y-%m-%d')
end_date = datetime.datetime.now().strftime('%Y-%m-%d')

fields = ["PX_LAST", "PX_VOLUME", "EQY_DVD_YLD_IND", "PX_TO_BOOK_RATIO",
          "BEST_PE_RATIO", "BEST_EPS", "SHORT_INT", "TOT_OPT_VOLUME_CUR_DAY"]
data = blp.bdh(tickers=['MSFT US Equity', 'AAPL US Equity', 'GOOG US Equity'] , flds=fields, start_date=start_date, end_date=end_date)
data = data.reset_index()
data

index MSFT US Equity                                               \
                       PX_LAST   PX_VOLUME EQY_DVD_YLD_IND PX_TO_BOOK_RATIO   
0    2023-03-27         276.38  26840212.0          0.9842          11.2387   
1    2023-03-28         275.23  21878647.0          0.9883          11.1919   
2    2023-03-29         280.51  25087032.0          0.9697          11.4066   
3    2023-03-30         284.05  25053410.0          0.9576          11.5505   
4    2023-03-31         288.30  32765976.0          0.9435          11.0132   
..          ...            ...         ...             ...              ...   
501  2025-03-18         383.52  19486850.0          0.8657           9.4203   
502  2025-03-19         387.82  19185532.0          0.8561           9.5259   
503  2025-03-20         386.84  18470542.0          0.8582           9.5018   
504  2025-03-21         391.26  39675928.0          0.8485           9.6104   
505  2025-03-24            NaN         NaN             NaN              NaN   

                                                              AAPL US Equity  \
    BEST_PE_RATIO BEST_EPS   SHORT_INT TOT_OPT_VOLUME_CUR_DAY        PX_LAST   
0          21.022    2.971         NaN               327120.0         158.28   
1          20.935    2.971         NaN               299519.0         157.65   
2          21.336      NaN         NaN               382466.0         160.77   
3          21.606      NaN         NaN               418178.0         162.36   
4          21.929      NaN  41113563.0               478752.0         164.90   
..            ...      ...         ...                    ...            ...   
501        27.603      NaN         NaN               158391.0         212.69   
502        27.915    3.226         NaN               169857.0         215.24   
503        27.844    3.226         NaN               296155.0         214.10   
504        28.162    3.226         NaN               300308.0         218.27   
505           NaN    3.226         NaN                    NaN            NaN   

     ...                                     GOOG US Equity              \
     ...    SHORT_INT TOT_OPT_VOLUME_CUR_DAY        PX_LAST   PX_VOLUME   
0    ...          NaN               723004.0         103.06  25393417.0   
1    ...          NaN               628584.0         101.36  24913480.0   
2    ...          NaN               721775.0         101.90  26148300.0   
3    ...          NaN               800028.0         101.32  25009829.0   
4    ...  111566048.0              1146552.0         104.00  28107953.0   
..   ...          ...                    ...            ...         ...   
501  ...          NaN               538309.0         162.67  24616784.0   
502  ...          NaN               882655.0         166.28  24955717.0   
503  ...          NaN              1037716.0         165.05  19981512.0   
504  ...          NaN               911708.0         166.25  29882064.0   
505  ...          NaN                    NaN            NaN         NaN   

                                                                         \
    EQY_DVD_YLD_IND PX_TO_BOOK_RATIO BEST_PE_RATIO BEST_EPS   SHORT_INT   
0               NaN              NaN           NaN      NaN         NaN   
1               NaN              NaN           NaN      NaN         NaN   
2               NaN              NaN           NaN      NaN         NaN   
3               NaN              NaN           NaN      NaN         NaN   
4               NaN              NaN           NaN      NaN  29937248.0   
..              ...              ...           ...      ...         ...   
501          0.4918              NaN        17.990    2.043         NaN   
502          0.4811              NaN        18.410    2.040         NaN   
503          0.4847              NaN        18.274      NaN         NaN   
504          0.4812              NaN        18.407    2.040         NaN   
505             NaN              NaN           NaN      NaN      

In [ ]:
start_date = (datetime.datetime.now() - datetime.timedelta(days=2*365)).strftime('%Y-%m-%d')
end_date = datetime.datetime.now().strftime('%Y-%m-%d')

fields = ["PX_LAST", "PX_VOLUME", "EQY_DVD_YLD_IND", "PX_TO_BOOK_RATIO",
          "BEST_PE_RATIO", "BEST_EPS", "SHORT_INT", "TOT_OPT_VOLUME_CUR_DAY"]
data = blp.bdh(tickers=['MSFT US Equity', 'AAPL US Equity', 'GOOG US Equity'] , flds=fields, start_date=start_date, end_date=end_date)
data

MSFT US Equity                                               \
                  PX_LAST   PX_VOLUME EQY_DVD_YLD_IND PX_TO_BOOK_RATIO   
2023-03-27         276.38  26840212.0          0.9842          11.2387   
2023-03-28         275.23  21878647.0          0.9883          11.1919   
2023-03-29         280.51  25087032.0          0.9697          11.4066   
2023-03-30         284.05  25053410.0          0.9576          11.5505   
2023-03-31         288.30  32765976.0          0.9435          11.0132   
...                   ...         ...             ...              ...   
2025-03-18         383.52  19486850.0          0.8657           9.4203   
2025-03-19         387.82  19185532.0          0.8561           9.5259   
2025-03-20         386.84  18470542.0          0.8582           9.5018   
2025-03-21         391.26  39675928.0          0.8485           9.6104   
2025-03-24            NaN         NaN             NaN              NaN   

                                                                      \
           BEST_PE_RATIO BEST_EPS   SHORT_INT TOT_OPT_VOLUME_CUR_DAY   
2023-03-27        21.022    2.971         NaN               327120.0   
2023-03-28        20.935    2.971         NaN               299519.0   
2023-03-29        21.336      NaN         NaN               382466.0   
2023-03-30        21.606      NaN         NaN               418178.0   
2023-03-31        21.929      NaN  41113563.0               478752.0   
...                  ...      ...         ...                    ...   
2025-03-18        27.603      NaN         NaN               158391.0   
2025-03-19        27.915    3.226         NaN               169857.0   
2025-03-20        27.844    3.226         NaN               296155.0   
2025-03-21        28.162    3.226         NaN               300308.0   
2025-03-24           NaN    3.226         NaN                    NaN   

           AAPL US Equity              ...               \
                  PX_LAST   PX_VOLUME  ...    SHORT_INT   
2023-03-27         158.28  52390266.0  ...          NaN   
2023-03-28         157.65  45992152.0  ...          NaN   
2023-03-29         160.77  51305691.0  ...          NaN   
2023-03-30         162.36  49501689.0  ...          NaN   
2023-03-31         164.90  68749792.0  ...  111566048.0   
...                   ...         ...  ...          ...   
2025-03-18         212.69  42432426.0  ...          NaN   
2025-03-19         215.24  54385391.0  ...          NaN   
2025-03-20         214.10  48862947.0  ...          NaN   
2025-03-21         218.27  94127768.0  ...          NaN   
2025-03-24            NaN         NaN  ...          NaN   

                                  GOOG US Equity                              \
           TOT_OPT_VOLUME_CUR_DAY        PX_LAST   PX_VOLUME EQY_DVD_YLD_IND   
2023-03-27               723004.0         103.06  25393417.0             NaN   
2023-03-28               628584.0         101.36  24913480.0             NaN   
2023-03-29               721775.0         101.90  26148300.0             NaN   
2023-03-30               800028.0         101.32  25009829.0             NaN   
2023-03-31              1146552.0         104.00  28107953.0             NaN   
...                           ...            ...         ...             ...   
2025-03-18               538309.0         162.67  24616784.0          0.4918   
2025-03-19               882655.0         166.28  24955717.0          0.4811   
2025-03-20              1037716.0         165.05  19981512.0          0.4847   
2025-03-21               911708.0         166.25  29882064.0          0.4812   
2025-03-24                    NaN            NaN         NaN             NaN   

                                                                \
           PX_TO_BOOK_RATIO BEST_PE_RATIO BEST_EPS   SHORT_INT   
2023-03-27              NaN           NaN      NaN         NaN   
2023-03-28              NaN           NaN      NaN         NaN   
2023-03-29              NaN           

In [ ]:
start_date = (datetime.datetime.now() - datetime.timedelta(days=2*365)).strftime('%Y-%m-%d')
end_date = datetime.datetime.now().strftime('%Y-%m-%d')

fields = ["PX_LAST", "PX_VOLUME", "EQY_DVD_YLD_IND", "PX_TO_BOOK_RATIO",
          "BEST_PE_RATIO", "BEST_EPS", "SHORT_INT", "TOT_OPT_VOLUME_CUR_DAY"]
data = blp.bdh(tickers=['MSFT US Equity', 'AAPL US Equity', 'GOOG US Equity'] , flds=fields, start_date=start_date, end_date=end_date)
data

MSFT US Equity                                               \
                  PX_LAST   PX_VOLUME EQY_DVD_YLD_IND PX_TO_BOOK_RATIO   
2023-03-27         276.38  26840212.0          0.9842          11.2387   
2023-03-28         275.23  21878647.0          0.9883          11.1919   
2023-03-29         280.51  25087032.0          0.9697          11.4066   
2023-03-30         284.05  25053410.0          0.9576          11.5505   
2023-03-31         288.30  32765976.0          0.9435          11.0132   
...                   ...         ...             ...              ...   
2025-03-18         383.52  19486850.0          0.8657           9.4203   
2025-03-19         387.82  19185532.0          0.8561           9.5259   
2025-03-20         386.84  18470542.0          0.8582           9.5018   
2025-03-21         391.26  39675928.0          0.8485           9.6104   
2025-03-24            NaN         NaN             NaN              NaN   

                                                                      \
           BEST_PE_RATIO BEST_EPS   SHORT_INT TOT_OPT_VOLUME_CUR_DAY   
2023-03-27        21.022    2.971         NaN               327120.0   
2023-03-28        20.935    2.971         NaN               299519.0   
2023-03-29        21.336      NaN         NaN               382466.0   
2023-03-30        21.606      NaN         NaN               418178.0   
2023-03-31        21.929      NaN  41113563.0               478752.0   
...                  ...      ...         ...                    ...   
2025-03-18        27.603      NaN         NaN               158391.0   
2025-03-19        27.915    3.226         NaN               169857.0   
2025-03-20        27.844    3.226         NaN               296155.0   
2025-03-21        28.162    3.226         NaN               300308.0   
2025-03-24           NaN    3.226         NaN                    NaN   

           AAPL US Equity              ...               \
                  PX_LAST   PX_VOLUME  ...    SHORT_INT   
2023-03-27         158.28  52390266.0  ...          NaN   
2023-03-28         157.65  45992152.0  ...          NaN   
2023-03-29         160.77  51305691.0  ...          NaN   
2023-03-30         162.36  49501689.0  ...          NaN   
2023-03-31         164.90  68749792.0  ...  111566048.0   
...                   ...         ...  ...          ...   
2025-03-18         212.69  42432426.0  ...          NaN   
2025-03-19         215.24  54385391.0  ...          NaN   
2025-03-20         214.10  48862947.0  ...          NaN   
2025-03-21         218.27  94127768.0  ...          NaN   
2025-03-24            NaN         NaN  ...          NaN   

                                  GOOG US Equity                              \
           TOT_OPT_VOLUME_CUR_DAY        PX_LAST   PX_VOLUME EQY_DVD_YLD_IND   
2023-03-27               723004.0         103.06  25393417.0             NaN   
2023-03-28               628584.0         101.36  24913480.0             NaN   
2023-03-29               721775.0         101.90  26148300.0             NaN   
2023-03-30               800028.0         101.32  25009829.0             NaN   
2023-03-31              1146552.0         104.00  28107953.0             NaN   
...                           ...            ...         ...             ...   
2025-03-18               538309.0         162.67  24616784.0          0.4918   
2025-03-19               882655.0         166.28  24955717.0          0.4811   
2025-03-20              1037716.0         165.05  19981512.0          0.4847   
2025-03-21               911708.0         166.25  29882064.0          0.4812   
2025-03-24                    NaN            NaN         NaN             NaN   

                                                                \
           PX_TO_BOOK_RATIO BEST_PE_RATIO BEST_EPS   SHORT_INT   
2023-03-27              NaN           NaN      NaN         NaN   
2023-03-28              NaN           NaN      NaN         NaN   
2023-03-29              NaN           

In [ ]:
# Get Bloomberg data
data = blp.bdh(tickers=['MSFT US Equity', 'AAPL US Equity', 'GOOG US Equity'], 
               flds=fields, 
               start_date=start_date, 
               end_date=end_date)

# Method 1: Stack the MultiIndex to create a "tidy" format
tidy_data = data.stack(level=0).reset_index()
# This will give columns: ['date', 'ticker', 'PX_LAST', 'PX_VOLUME', ...]

# Method 2: Access specific ticker-field combinations
msft_price = data[('MSFT US Equity', 'PX_LAST')]
aapl_price = data[('AAPL US Equity', 'PX_LAST')]

# Method 3: Flatten column names with meaningful names
data.columns = [f"{ticker}_{field}" for ticker, field in data.columns]

# Example usage:
print("Method 1 - Tidy format:")
print(tidy_data.head())

print("\nMethod 2 - Direct access:")
print(msft_price.head())

print("\nMethod 3 - Flattened columns:")
print(data.columns)

Method 1 - Tidy format:
      level_0         level_1  PX_LAST   PX_VOLUME  EQY_DVD_YLD_IND  \
0  2023-03-27  AAPL US Equity   158.28  52390266.0           0.5812   
1  2023-03-27  GOOG US Equity   103.06  25393417.0              NaN   
2  2023-03-27  MSFT US Equity   276.38  26840212.0           0.9842   
3  2023-03-28  AAPL US Equity   157.65  45992152.0           0.5836   
4  2023-03-28  GOOG US Equity   101.36  24913480.0              NaN   

   PX_TO_BOOK_RATIO  BEST_PE_RATIO  BEST_EPS  SHORT_INT  \
0           44.2036         21.412       NaN        NaN   
1               NaN            NaN       NaN        NaN   
2           11.2387         21.022     2.971        NaN   
3           44.0276         21.327       NaN        NaN   
4               NaN            NaN       NaN        NaN   

   TOT_OPT_VOLUME_CUR_DAY  
0                723004.0  
1                238130.0  
2                327120.0  
3                628584.0  
4                201364.0  

Method 2 - Direct access:

C:\Users\luigi\AppData\Local\Temp\ipykernel_35600\3690810507.py:8: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  tidy_data = data.stack(level=0).reset_index()
